<a href="https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content lifecycle: growing vs declining

The research report observed that growing pages were younger on average than declining pages, with average ages of 185 days and 228 days respectively. It also reported that average word count was almost the same between the two groups.

**Methodology question:** The trend direction is based on comparing the most recent 30 days with the previous 30 days. Does this label support a claim about sustained growth or decline, or is it more appropriate to describe it as a short-term directional signal? I would describe it as an observed short-term relationship unless longer-term validation is available.

### Finding 2 — The CTR cliff

The report observed that weighted CTR was highest for pages ranking in the top three positions and decreased for pages ranking lower in search results.

**Methodology question:** This is an observed relationship between search position and CTR. Does the evidence support a causal claim that moving a page higher will cause the same CTR increase, or should it be described as an observed relationship that may also depend on query mix, search intent, and other factors? I would describe it as measured observational evidence rather than proof of causation.

I will apply the same careful standard to my own model and describe its results as observed, measured, and directional decision support rather than certainty or causation.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I compare two validation designs using the same Week-5 model features and target.

- **Before:** a stratified row split. Rows from the same client may appear in both training and test data.
- **After:** a client-holdout split. All rows belonging to selected clients are kept in the test set, so those clients are not present in training.

The client-holdout split is a more honest estimate when the intended use is on unseen clients because it checks whether the model transfers beyond clients represented during training.

I will report the measured before-and-after results without overstating them.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 — Section 2: Before/after validation comparison

import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# ---------------------------------------------------------
# 1. Repository
# ---------------------------------------------------------

repo_dir = Path("/content/flyrank-ml-internship")

if not repo_dir.exists():
    raise FileNotFoundError(
        "FlyRank repository was not found."
    )

os.chdir(repo_dir)

print("PASS: Repository found")
print("Path:", repo_dir)


# ---------------------------------------------------------
# 2. Dataset
# ---------------------------------------------------------

data_path = (
    repo_dir
    / "data"
    / "raw"
    / "content_refresh_anonymized.csv"
)

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {data_path}"
    )

print("PASS: Dataset found")


# ---------------------------------------------------------
# 3. Prepare Week-5 features
# ---------------------------------------------------------

prepare_script = (
    repo_dir
    / "scripts"
    / "01_prepare_features.py"
)

if prepare_script.exists():

    subprocess.run(
        [sys.executable, str(prepare_script)],
        check=True
    )

    print("PASS: Week-5 features prepared")

else:

    print(
        "Feature preparation script not found. "
        "Using existing processed features."
    )


feature_path = (
    repo_dir
    / "data"
    / "processed"
    / "refresh_feature_vector.csv"
)

if not feature_path.exists():
    raise FileNotFoundError(
        f"Feature file not found: {feature_path}"
    )

print("PASS: Feature file found")


# ---------------------------------------------------------
# 4. Official feature definitions
# ---------------------------------------------------------

scripts_dir = repo_dir / "scripts"

if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

from ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES
)

print("PASS: Feature definitions loaded")


# ---------------------------------------------------------
# 5. Load data
# ---------------------------------------------------------

frame = pd.read_csv(feature_path)

required_columns = [
    "content_id",
    "client_id",
    "is_declining_label"
]

missing_columns = [
    column
    for column in required_columns
    if column not in frame.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Rows:", len(frame))


# ---------------------------------------------------------
# 6. Build numeric features
# ---------------------------------------------------------

numeric_features = [
    column
    for column in MODEL_NUMERIC_FEATURES
    if column in frame.columns
]

numeric_frame = (
    frame[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)


# ---------------------------------------------------------
# 7. Build categorical features
# ---------------------------------------------------------

categorical_features = [
    column
    for column in MODEL_CATEGORICAL_FEATURES
    if column in frame.columns
]

if categorical_features:

    categorical_frame = (
        frame[categorical_features]
        .fillna("unknown")
        .astype(str)
    )

    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dtype=float
    )

else:

    encoded_frame = pd.DataFrame(
        index=frame.index
    )


# ---------------------------------------------------------
# 8. Final X and y
# ---------------------------------------------------------

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_frame.reset_index(drop=True)
    ],
    axis=1
)

y = (
    frame["is_declining_label"]
    .astype(int)
    .reset_index(drop=True)
)

if y.nunique() != 2:
    raise ValueError(
        "The target must contain exactly two classes."
    )

print("Model features:", X.shape[1])
print(
    "Declining rate:",
    round(float(y.mean()), 4)
)


# ---------------------------------------------------------
# 9. Model
# ---------------------------------------------------------

def make_model():

    return Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler()
            ),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=42
                )
            )
        ]
    )


# ---------------------------------------------------------
# 10. Evaluation
# ---------------------------------------------------------

def evaluate_split(
    train_idx,
    test_idx,
    split_name
):

    model = make_model()

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model.fit(
        X_train,
        y_train
    )

    probabilities = (
        model
        .predict_proba(X_test)[:, 1]
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    return {
        "split": split_name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "accuracy": accuracy_score(
            y_test,
            predictions
        ),
        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        ),
        "average_precision":
            average_precision_score(
                y_test,
                probabilities
            )
    }


# ---------------------------------------------------------
# 11. BEFORE — row split
# ---------------------------------------------------------

all_indices = np.arange(
    len(frame)
)

random_train, random_test = (
    train_test_split(
        all_indices,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

before = evaluate_split(
    random_train,
    random_test,
    "Before: stratified row split"
)


# ---------------------------------------------------------
# 12. AFTER — client holdout
# ---------------------------------------------------------

client_series = (
    frame["client_id"]
    .fillna("unknown")
    .astype(str)
)

unique_clients = (
    client_series
    .drop_duplicates()
    .to_numpy()
)

if len(unique_clients) < 2:
    raise ValueError(
        "At least two unique clients are required."
    )


rng = np.random.default_rng(42)

successful_split = False

for attempt in range(500):

    shuffled_clients = (
        rng.permutation(unique_clients)
    )

    test_client_count = max(
        1,
        int(
            round(
                len(unique_clients) * 0.20
            )
        )
    )

    test_client_count = min(
        test_client_count,
        len(unique_clients) - 1
    )

    test_clients = set(
        shuffled_clients[
            :test_client_count
        ]
    )

    client_test_mask = (
        client_series
        .isin(test_clients)
        .to_numpy()
    )

    client_train = all_indices[
        ~client_test_mask
    ]

    client_test = all_indices[
        client_test_mask
    ]

    if (
        len(client_train) > 0
        and len(client_test) > 0
        and y.iloc[client_train].nunique() == 2
        and y.iloc[client_test].nunique() == 2
    ):

        successful_split = True
        break


if not successful_split:

    raise ValueError(
        "Could not create a client-holdout split "
        "containing both target classes."
    )


after = evaluate_split(
    client_train,
    client_test,
    "After: client-holdout split"
)


# ---------------------------------------------------------
# 13. Comparison
# ---------------------------------------------------------

comparison = pd.DataFrame(
    [before, after]
)

print("\nVALIDATION COMPARISON")
print("=" * 100)

display(
    comparison.round(4)
)


# ---------------------------------------------------------
# 14. Client leakage check
# ---------------------------------------------------------

train_clients = set(
    client_series.iloc[client_train]
)

test_clients = set(
    client_series.iloc[client_test]
)

shared_clients = (
    train_clients & test_clients
)

print(
    "\nShared clients:",
    len(shared_clients)
)

if shared_clients:
    raise ValueError(
        "Client leakage detected."
    )

print(
    "PASS: train and test clients are separate."
)

print(
    "\nSection 2 completed successfully."
)



PASS: Repository found
Path: /content/flyrank-ml-internship
PASS: Dataset found
PASS: Week-5 features prepared
PASS: Feature file found
PASS: Feature definitions loaded
Rows: 30000
Model features: 52
Declining rate: 0.5421

VALIDATION COMPARISON


,split,train_rows,test_rows,accuracy,precision,recall,f1,roc_auc,average_precision
0,Before: stratified row split,24000,6000,0.6502,0.6775,0.6765,0.6770,0.7107,0.7271
1,After: client-holdout split,27675,2325,0.6606,0.5659,0.5666,0.5662,0.7003,0.5215



Shared clients: 0
PASS: train and test clients are separate.

Section 2 completed successfully.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audit the final model feature set for target leakage, future information, and product or action flags that could reveal the answer.

The target is `is_declining_label`, so the target itself and fields used to define the target should not be model features.

I also inspect anonymized failures from the honest client-holdout test set. These examples show where the model can make mistakes and support treating the model as directional decision support rather than certainty.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-09 — Section 3: Leakage audit and failure examples


# ---------------------------------------------------------
# 1. Leakage audit
# ---------------------------------------------------------

all_model_features = (
    list(MODEL_NUMERIC_FEATURES)
    + list(MODEL_CATEGORICAL_FEATURES)
)

forbidden_exact = {
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate"
}


leakage_hits = sorted(
    set(all_model_features)
    .intersection(forbidden_exact)
)


print("LEAKAGE AUDIT")
print("=" * 80)

print(
    "Target:",
    "is_declining_label"
)

print(
    "Target used as feature:",
    "is_declining_label"
    in all_model_features
)

print(
    "Forbidden intersections:",
    leakage_hits
)


if "is_declining_label" in all_model_features:
    raise ValueError(
        "Leakage detected: target is a feature."
    )

if leakage_hits:
    raise ValueError(
        f"Forbidden leakage features found: "
        f"{leakage_hits}"
    )


print(
    "\nPASS: no forbidden target or "
    "product/action leakage was found."
)


# ---------------------------------------------------------
# 2. Train honest model
# ---------------------------------------------------------

honest_model = Pipeline(
    steps=[
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


honest_model.fit(
    X.iloc[client_train],
    y.iloc[client_train]
)


honest_probabilities = (
    honest_model
    .predict_proba(
        X.iloc[client_test]
    )[:, 1]
)


honest_predictions = (
    honest_probabilities >= 0.5
).astype(int)


# ---------------------------------------------------------
# 3. Failure examples
# ---------------------------------------------------------

test_rows = (
    frame.iloc[client_test]
    .copy()
    .reset_index(drop=True)
)


test_rows["actual"] = (
    y.iloc[client_test]
    .to_numpy()
)


test_rows["predicted"] = (
    honest_predictions
)


test_rows["probability"] = (
    honest_probabilities
)


errors = test_rows[
    test_rows["actual"]
    != test_rows["predicted"]
].copy()


safe_columns = [
    column
    for column in [
        "content_age_days",
        "days_since_last_update",
        "ctr",
        "avg_position",
        "engagement_rate",
        "scroll_rate",
        "ai_traffic_pct",
        "actual",
        "predicted",
        "probability"
    ]
    if column in errors.columns
]


print("\nFAILURE ANALYSIS")
print("=" * 80)

print(
    "Misclassified test rows:",
    len(errors)
)


if errors.empty:

    print(
        "No misclassified rows were found."
    )

else:

    display(
        errors[
            safe_columns
        ]
        .head(10)
        .round(4)
    )


print("\nINTERPRETATION")

print(
    "The failure examples show that the model "
    "can make mistakes on unseen clients."
)

print(
    "Therefore, the model should be treated "
    "as directional decision support rather "
    "than certainty about future page decline."
)

print(
    "\nSection 3 completed successfully."
)


LEAKAGE AUDIT
Target: is_declining_label
Target used as feature: False
Forbidden intersections: []

PASS: no forbidden target or product/action leakage was found.

FAILURE ANALYSIS
Misclassified test rows: 789


,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,actual,predicted,probability
1,91,1,0.00,8.3,0.00,50.00,0.0,1,0,0.4729
11,91,8,0.00,3.6,0.00,0.00,0.0,1,0,0.4210
15,144,20,0.11,6.4,0.00,2.56,0.0,0,1,0.7039
17,175,20,0.00,10.2,9.09,9.09,0.0,0,1,0.8279
18,92,20,0.31,5.3,2.04,1.82,0.0,0,1,0.5495
20,125,20,0.00,2.4,0.00,0.00,0.0,1,0,0.4799
23,116,8,16.67,9.3,0.00,100.00,0.0,1,0,0.2192
24,144,20,0.28,12.3,0.00,0.00,0.0,1,0,0.0003
27,489,20,12.50,2.6,0.00,100.00,0.0,1,0,0.1252
28,175,20,0.18,29.0,0.00,0.00,0.0,0,1,0.5714



INTERPRETATION
The failure examples show that the model can make mistakes on unseen clients.
Therefore, the model should be treated as directional decision support rather than certainty about future page decline.

Section 3 completed successfully.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Safe claim

The model provides observed and measured directional signals for identifying pages that may show declining performance.

The client-holdout result is a more honest estimate when the intended use is on unseen clients.

The results are measured on this dataset and should not be interpreted as proof of causation or guaranteed future performance.

The model should be used as directional decision support and reviewed before taking a content action.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



## Self-check

- [x] Section 1 contains two research findings and constructive methodology questions.
- [x] Section 2 compares a normal row split with a client-holdout split.
- [x] Section 2 checks that train and test clients do not overlap.
- [x] Section 3 audits the final feature set for leakage.
- [x] Section 3 includes anonymized model failure examples.
- [x] Section 4 uses observed, measured, directional, and decision-support language.
- [ ] Notebook runs top to bottom without errors.
- [ ] Notebook is committed under `work/notebooks/w06_validation_audit.ipynb`.
- [ ] Repository URL is submitted on the ML-09 assignment card.